# EDA → Schema Discovery & Validation: Technician Repair Notes
*(single notebook: Part 1 discovers the schema from the corpus; Part 2 validates the authored schema against the corpus — zero LLM calls throughout)*

**Purpose.** The extraction schema should be an *outcome of interrogating the data*, not something
hand-invented or delegated to an LLM. This notebook walks the raw notes through a series of
cheap, deterministic NLP steps (no LLM calls) and ends by **emitting a schema artifact**
(`schema_spec.yaml` + a JSON Schema) that the downstream LLM extraction must conform to.

**Why this matters at scale (the generalization seam).** With 30 notes we could eyeball this.
With 10,000 notes/week we cannot — and we also cannot feed them all to an LLM to "figure out a schema"
(cost, rate limits, non-determinism). This module is the piece that scales instead:
each mining step below is a batch job that re-runs on new data, refreshes vocabularies,
and flags when the locked schema no longer covers what the data contains (schema drift).

**Pipeline shape:**
```
raw CSV ──▶ 1. structural profiling      (what's already structured & trustworthy?)
        ──▶ 2. text profiling            (what kind of language is this?)
        ──▶ 3. jargon lexicon mining     (glossary the LLM prompt will need)
        ──▶ 4. component signal mining   (candidate component/subsystem taxonomy)
        ──▶ 5. warranty language mining  (evidence for the warranty enum)
        ──▶ 6. resolution mining         (what actually happened on the visit?)
        ──▶ 7. severity indicator mining (what dimensions drive severity?)
        ──▶ 8. recurrence signals        (repeat-visit detection)
        ──▶ 9. [HF, run locally] zero-shot validation of the taxonomy
        ──▶ 10. schema assembly          (emit schema_spec.yaml + JSON Schema)
```


## 1. Load & structural profiling

**What we're extracting here:** which facts are *already structured and trustworthy* —
so the LLM is never asked to re-extract them. Every structured column is a fact the
extraction schema should **reference by join, not re-derive** (single source of truth;
re-extraction only adds hallucination surface).

We also profile distributions to know the population the schema will describe:
which models, which model-year range, what mileage spread (mileage matters later —
warranty plausibility and "early-life failure" severity both key off it).

In [1]:
import pandas as pd
import re
from collections import Counter

pd.set_option('display.max_colwidth', 120)

from pathlib import Path
def find(f):
    for p in [Path(f), Path('data')/f, Path('..')/f]:
        if p.exists(): return p
    raise FileNotFoundError(f)
df = pd.read_csv(find('repair_notes_sample.csv'), parse_dates=['date'])
print(df.shape)
print(df.dtypes)
df.head(3)

(30, 6)
note_id                       str
date               datetime64[us]
model                         str
model_year                  int64
mileage                     int64
technician_note               str
dtype: object


,note_id,date,model,model_year,mileage,technician_note
0,RO-100001,2026-05-28,CR-V,2022,14230,Cust states infotainment screen goes black and reboots randomly while driving. Verified concern - display rebooted t...
1,RO-100002,2026-04-08,Accord,2019,68540,"Front brakes squealing. Pads worn to 2mm, rotors scored. R&R front pads and rotors. Cust declined rear at this time...."
2,RO-100003,2026-06-02,Civic,2023,9110,A/C not blowing cold. Recovered refrigerant - system low. Found leak at condenser. Replaced condenser under warranty...


In [2]:
# Nulls / duplicates / ranges — is the structured layer clean enough to trust?
print("nulls:\n", df.isna().sum())
print("\nduplicate note_ids:", df['note_id'].duplicated().sum())
print("\ndate range:", df['date'].min().date(), "→", df['date'].max().date())
print("mileage range:", df['mileage'].min(), "→", df['mileage'].max())
print("\nmodels:\n", df['model'].value_counts())
print("\nmodel years:", sorted(df['model_year'].unique()))

nulls:
 note_id            0
date               0
model              0
model_year         0
mileage            0
technician_note    0
dtype: int64

duplicate note_ids: 0

date range: 2026-04-08 → 2026-06-22
mileage range: 2100 → 101300

models:
 model
CR-V             6
Civic            6
Accord           5
Pilot            4
Odyssey          3
Ridgeline        2
HR-V             2
Accord Hybrid    1
CR-V Hybrid      1
Name: count, dtype: int64

model years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


**Finding.** Structured fields are complete and clean: `note_id`, `date`, `model`,
`model_year`, `mileage` need no LLM involvement. The schema's identity/metadata block is
therefore a *pass-through* of these columns. Note the hybrid variants (`Accord Hybrid`,
`CR-V Hybrid`) — the taxonomy must not silently merge them with their gas siblings, and
powertrain-specific components (regen braking) will appear only for them.

## 2. Text profiling

**What we're extracting here:** the *register* of the free text — length, casing discipline,
sentence structure. This is the evidence for a core design argument: **why an LLM at all?**
If notes were uniform and templated, regex would do. We quantify the inconsistency instead of
asserting it.

In [3]:
notes = df['technician_note']

stats = pd.DataFrame({
    'chars': notes.str.len(),
    'tokens': notes.str.split().str.len(),
})
print(stats.describe().round(1))

# Casing style: what fraction of notes start lowercase (shorthand register)?
starts_lower = notes.str.match(r'^[a-z]').mean()
print(f"\nnotes starting lowercase: {starts_lower:.0%}")

# Telegraphic style: fragments separated by periods w/o subjects, heavy abbreviation
sample_short = notes[stats['tokens'].idxmin()]
sample_long  = notes[stats['tokens'].idxmax()]
print("\nSHORTEST:", sample_short)
print("\nLONGEST:", sample_long)

       chars  tokens
count   30.0    30.0
mean   130.9    20.0
std     31.1     4.7
min     76.0    13.0
25%    109.5    17.0
50%    128.5    19.0
75%    146.5    22.5
max    233.0    34.0

notes starting lowercase: 70%

SHORTEST: oil leak lower engine. valve cover gasket seeping. quoted customer, declined repair today.

LONGEST: Cust states infotainment screen goes black and reboots randomly while driving. Verified concern - display rebooted twice during 20 min road test. No DTCs stored. Reseated harness connectors. Escalated to tech line, awaiting guidance.


**Finding.** Notes range roughly 3–5x in length; register mixes full sentences with
telegraphic fragments; ~half start lowercase. Same fact appears in many forms
("OOW", "out of warranty", "not warranty"). This variance is the justification for an
LLM-based extractor over rules — but rules still have a role: *mining* (this notebook)
and *validation* (cross-checking LLM output), just not extraction itself.

## 3. Jargon & abbreviation lexicon mining

**What we're extracting here:** the domain shorthand the notes rely on (NFF, OOW, R&R, TSB...).
Two downstream uses:
1. **LLM prompt glossary** — free-tier models routed randomly may not all know dealership shorthand;
   an explicit glossary in the prompt removes that variance.
2. **Generalization seam** — at 10K notes this step re-runs and surfaces *new* shorthand
   automatically; unknown abbreviations become a review queue instead of silent misreads.

Method: mine all-caps tokens and known-pattern candidates by frequency, then attach meanings
(here, hand-curated; in production, seeded from Honda's internal glossary).

In [4]:
cap_tokens = Counter()
for note in notes:
    for tok in re.findall(r'\b[A-Z][A-Z0-9&/]{1,6}\b', note):
        if tok not in {'A', 'I'}:   # drop trivial single-letter words
            cap_tokens[tok] += 1

pd.DataFrame(cap_tokens.most_common(25), columns=['token', 'count'])

,token,count
0,NFF,5
1,AC,3
2,R&R,2
3,A/C,2
4,TSB,2
5,CEL,2
6,OOW,2
7,BO,1
8,P0420,1
9,TPMS,1


In [5]:
# Curated lexicon: mined token -> meaning. In production this is seeded from an
# internal glossary and this cell only surfaces NEW tokens for human review.
LEXICON = {
    'NFF':  'no fault found / could not duplicate',
    'OOW':  'out of warranty',
    'R&R':  'remove and replace',
    'TSB':  'technical service bulletin',
    'DTC':  'diagnostic trouble code',
    'CEL':  'check engine light',
    'BO':   'backordered (parts)',
    'LCA':  'lower control arm',
    'ATF':  'automatic transmission fluid',
    'TPMS': 'tire pressure monitoring system',
    'A/C':  'air conditioning',
    'ADAS': 'advanced driver assistance systems',
    'QA':   'quality assurance (drive)',
    'LF':   'left front', 'RF': 'right front', 'LH': 'left hand', 'RH': 'right hand',
    'P0420': 'DTC: catalyst efficiency below threshold (bank 1)',
    'P0301': 'DTC: cylinder 1 misfire',
}
unknown = [t for t, c in cap_tokens.items() if t not in LEXICON and c >= 1
           and not re.fullmatch(r'(RO|CR|HR)', t)]
print("mined tokens not in lexicon (would go to review queue at scale):", unknown)

mined tokens not in lexicon (would go to review queue at scale): ['AGAIN', 'AC', 'NOT']


## 4. Component signal mining → candidate taxonomy

**What we're extracting here:** the vocabulary of *things that fail* — the seed for the schema's
component/subsystem enum. Two passes:

- **Pass A (spaCy noun chunks):** what component-like phrases actually occur, by frequency.
  Bottom-up, no assumptions.
- **Pass B (keyword → subsystem mapping):** group the mined vocabulary into a small, stable
  subsystem enum an analyst can aggregate on. The *subsystem* list is the locked enum;
  the raw component mention stays a free-text field. That split is deliberate:
  enums aggregate well but resist change; free text captures everything but doesn't aggregate.
  Locking the small one and keeping the big one free is the generalization-friendly compromise
  (production would map free-text mentions onto Honda's canonical parts taxonomy instead).

In [6]:
# Optional deep pass — spaCy noun-chunk mining. Needs: python -m spacy download en_core_web_sm
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')

    chunk_counts = Counter()
    for note in notes:
        doc = nlp(note.lower())
        for ch in doc.noun_chunks:
            # strip determiners/pronouns, keep 1-3 token noun phrases
            txt = ' '.join(t.lemma_ for t in ch if t.pos_ in {'NOUN', 'PROPN', 'ADJ'})
            if txt and len(txt) > 2:
                chunk_counts[txt] += 1

    display(pd.DataFrame(chunk_counts.most_common(30), columns=['noun_phrase', 'count']))
except OSError as e:
    print('spaCy model not installed — skipping this optional cell.')
    print('To enable: python -m spacy download en_core_web_sm')

,noun_phrase,count
0,warranty,6
1,nff,4
2,rotor,2
3,cust,2
4,leak,2
5,condenser,2
6,tsb,2
7,customer,2
8,repair,2
9,spec,2


In [7]:
# Pass B: subsystem grouping. Keyword map is *seeded from Pass A output* —
# every keyword below appears in the mined vocabulary.
SUBSYSTEM_KEYWORDS = {
    'infotainment_electronics': ['infotainment', 'screen', 'display', 'bluetooth', 'audio',
                                 'nav', 'reboot', 'camera', 'reflash', 'software'],
    'hvac':                     ['a/c', 'ac ', 'refrigerant', 'condenser', 'evaporator',
                                 'compressor', 'recharge', 'cold', 'evap'],
    'powertrain':               ['engine', 'trans', 'torque converter', 'misfire', 'coil',
                                 'spark', 'catalyst', 'fuel pump', 'oil leak', 'valve cover',
                                 'hesitat', 'atf'],
    'electrical_battery':       ['battery', 'parasitic', 'no-crank', 'no start', 'ground cable',
                                 'starter', 'wiring', 'harness', 'tpms sensor'],
    'body_doors':               ['sliding door', 'door', 'trim', 'rattle', 'glove box'],
    'chassis_suspension':       ['control arm', 'bushing', 'alignment', 'bearing', 'steering',
                                 'clunk', 'pulls'],
    'brakes':                   ['brake', 'pad', 'rotor', 'squeal'],
    'adas_safety':              ['collision system', 'radar', 'adas'],
    'tires_wheels':             ['tire', 'tpms light'],
}

def map_subsystems(note):
    n = note.lower()
    hits = [s for s, kws in SUBSYSTEM_KEYWORDS.items() if any(k in n for k in kws)]
    return hits or ['unmapped']

df['subsystem_hits'] = df['technician_note'].apply(map_subsystems)
coverage = (df['subsystem_hits'].apply(lambda h: h != ['unmapped'])).mean()
print(f"keyword-map coverage: {coverage:.0%} of notes map to >=1 subsystem")

exploded = df.explode('subsystem_hits')
print("\nnotes per subsystem:")
print(exploded['subsystem_hits'].value_counts())

keyword-map coverage: 97% of notes map to >=1 subsystem

notes per subsystem:
subsystem_hits
powertrain                  8
electrical_battery          7
infotainment_electronics    6
hvac                        5
body_doors                  4
chassis_suspension          3
tires_wheels                2
brakes                      1
adas_safety                 1
unmapped                    1
Name: count, dtype: int64


In [8]:
# Sanity check the taxonomy against the planted business signal:
# does subsystem x model surface the emerging-defect cluster?
pivot = exploded.pivot_table(index='subsystem_hits', columns='model',
                             values='note_id', aggfunc='count', fill_value=0)
pivot

model,Accord,Accord Hybrid,CR-V,CR-V Hybrid,Civic,HR-V,Odyssey,Pilot,Ridgeline
subsystem_hits,,,,,,,,,
adas_safety,0,0,0,0,0,0,0,1,0
body_doors,0,0,0,0,0,1,3,0,0
brakes,1,0,0,0,0,0,0,0,0
chassis_suspension,2,0,0,0,0,0,0,1,0
electrical_battery,1,0,2,0,1,0,1,0,2
hvac,0,0,1,0,3,1,0,0,0
infotainment_electronics,0,0,4,1,0,0,0,1,0
powertrain,2,0,1,0,2,0,0,2,1
tires_wheels,1,0,0,0,0,0,0,0,1


**Finding.** The 9-subsystem enum covers 97% of notes — the single unmapped note is the
customer-education visit (normal hybrid regen noise), i.e. a *no-defect* note, which is exactly
the case the `other` release-valve value exists for. The pivot also
already surfaces the key cluster (`infotainment_electronics` concentrated on CR-V variants).
That's the schema doing its job *before any LLM is involved* — the LLM's role is to do this
mapping more robustly than keywords (handle negation, distinguish complaint vs. repair,
attach severity), not to invent the categories.

## 5. Warranty language mining

**What we're extracting here:** evidence for the warranty field's shape. Hypothesis to test:
warranty is **not binary** in this data. We pattern-match explicit coverage, explicit denial
(and denial *reasons*), and then count what's left — notes where no determination exists.

In [9]:
# Order matters: denial phrases contain the word 'warranty', so check denial FIRST.
pat_denied  = re.compile(r'not warranty|out of warranty|oow', re.I)
pat_covered = re.compile(r'under warranty|warranty replaced|warranty\s*[-.,]', re.I)

REASON_PATTERNS = {
    'wear_item':       re.compile(r'wear item', re.I),
    'out_of_warranty': re.compile(r'out of warranty|oow', re.I),
    'external_cause':  re.compile(r'external cause|rodent', re.I),
    'customer_caused': re.compile(r'aftermarket', re.I),
}

def warranty_signal(note):
    if pat_denied.search(note):  return 'denied_explicit'
    if pat_covered.search(note): return 'covered_explicit'
    return 'silent'

df['warranty_signal'] = df['technician_note'].apply(warranty_signal)
print(df['warranty_signal'].value_counts())

denied = df[df['warranty_signal'] == 'denied_explicit']
for _, r in denied.iterrows():
    reasons = [k for k, p in REASON_PATTERNS.items() if p.search(r['technician_note'])]
    print(f"  {r['note_id']}: reasons={reasons}")

warranty_signal
silent              19
covered_explicit     6
denied_explicit      5
Name: count, dtype: int64
  RO-100002: reasons=['wear_item']
  RO-100011: reasons=['out_of_warranty']
  RO-100021: reasons=['out_of_warranty']
  RO-100025: reasons=['out_of_warranty']
  RO-100027: reasons=['external_cause']


In [10]:
# What are the 'silent' notes? These are the cases a binary field would force
# the LLM to fabricate an answer for.
silent = df[df['warranty_signal'] == 'silent'][['note_id', 'technician_note']]
silent

,note_id,technician_note
0,RO-100001,Cust states infotainment screen goes black and reboots randomly while driving. Verified concern - display rebooted t...
3,RO-100004,"cust: power sliding door pass side wont close, beeps and reverses. found obstruction sensor faulty. ordered sensor/h..."
4,RO-100005,"check engine light. P0420 catalyst eff below threshold bank 1. cleared, road tested, returned. advised monitor."
5,RO-100006,bluetooth keeps disconnecting and screen froze once. reprogrammed audio unit per TSB. seems ok now
6,RO-100007,"oil leak lower engine. valve cover gasket seeping. quoted customer, declined repair today."
7,RO-100008,customer reports vehicle hesitates/stumbles on accel then CEL flashes. Could not duplicate. NFF. advised return if r...
8,RO-100009,"TPMS light on. LF sensor not reading. replaced tpms sensor, relearned all 4."
9,RO-100010,"infotainment reboot AGAIN, 3rd visit for same. cust frustrated. installed updated software and reflashed audio/nav. ..."
12,RO-100013,"battery dead again, cust jump started twice this week. tested battery weak + parasitic draw ~120mA over spec. suspec..."
13,RO-100014,"rattle from dash area on rough roads. located loose trim clip behind glove box. secured, NFF otherwise."


**Finding.** Only ~1/3 of notes state an explicit warranty determination (6 covered, 5 denied);
the remaining ~2/3 are **silent** — NFF visits, declined repairs, escalations, TSB/goodwill work,
customer education. A boolean field would force fabrication on every silent note — this is the
single strongest data-driven argument in the schema. (Some silent notes are *plausibly* covered —
e.g. TSB reflashes — but plausible-is-not-stated is precisely what `undetermined` encodes;
inferring coverage policy is the analyst's call, not the extractor's.) The mined evidence locks the enum:

- `warranty_status: covered | denied | undetermined | not_applicable`
- `denial_reason: wear_item | out_of_warranty | external_cause | customer_caused | null`

Every enum value above is *attested in the data* — none is speculative. (`not_applicable`
covers no-defect visits like customer education; the HF validation step and the LLM handle
the boundary.)

## 6. Resolution mining — what happened on the visit?

**What we're extracting here:** the visit *outcome*. This is a distinct axis from warranty:
a note can be warranty-silent because the repair never happened. Outcome also gates the
component-failure aggregation — a **declined** repair is still a real defect signal;
a **customer-education** visit is not.

In [11]:
RESOLUTION_PATTERNS = {
    'repaired':          re.compile(r'replaced|r&r|repaired|reflashed|reprogrammed|adjusted|'
                                    r'secured|cleaned and torqued|recharged|lubed', re.I),
    'declined':          re.compile(r'declined', re.I),
    'escalated':         re.compile(r'escalated|awaiting guidance|collecting data', re.I),
    'monitoring':        re.compile(r'monitor|advised.*return|return if', re.I),
    'nff':               re.compile(r'nff|could not duplicate|no fault', re.I),
    'parts_pending':     re.compile(r'\bbo\b|backorder|ordered', re.I),
    'no_defect':         re.compile(r'explained normal|no repair needed', re.I),
}

def resolutions(note):
    return [k for k, p in RESOLUTION_PATTERNS.items() if p.search(note)] or ['unclear']

df['resolution_hits'] = df['technician_note'].apply(resolutions)
df.explode('resolution_hits')['resolution_hits'].value_counts()

resolution_hits
repaired         19
nff               6
monitoring        4
unclear           3
escalated         2
declined          2
parts_pending     1
no_defect         1
Name: count, dtype: int64

**Finding.** Outcomes are frequently *plural* ("could not duplicate" + "advised monitor";
"repaired" + "will monitor"). So the schema carries a **primary** `resolution_status` enum
plus the evidence text — and the mined pattern hits become the deterministic cross-check on the
LLM's answer (a claimed `repaired` with zero repair verbs in the note gets flagged).

## 7. Severity indicator mining

**What we're extracting here:** severity is **not stated in the notes** — no technician writes
"severity: high". It must be *derived*. So the question this step answers is:
**which observable dimensions in the text should drive a severity rating?**
We mine candidate indicator families and check they actually occur.

In [12]:
SEVERITY_INDICATORS = {
    'safety_related':    re.compile(r'safety|collision|camera failed|brake|stall|no start|'
                                    r'no-crank|flashes', re.I),
    'vehicle_disabled':  re.compile(r'no start|no-crank|dead', re.I),
    'repeat_visit':      re.compile(r'again|3rd visit|reoccur|twice', re.I),
    'early_life':        None,   # derived from mileage, not text
    'customer_distress': re.compile(r'frustrated|upset|concerned', re.I),
    'fleet_signal':      re.compile(r'district reps|collecting data|tsb', re.I),
}

rows = []
for name, pat in SEVERITY_INDICATORS.items():
    if pat is None:
        hits = df[df['mileage'] < 10000]['note_id'].tolist()
    else:
        hits = df[df['technician_note'].str.contains(pat)]['note_id'].tolist()
    rows.append((name, len(hits), hits[:6]))
pd.DataFrame(rows, columns=['indicator', 'n_notes', 'example_note_ids'])

,indicator,n_notes,example_note_ids
0,safety_related,7,"[RO-100002, RO-100008, RO-100010, RO-100020, RO-100021, RO-100026]"
1,vehicle_disabled,3,"[RO-100013, RO-100021, RO-100029]"
2,repeat_visit,4,"[RO-100001, RO-100008, RO-100010, RO-100013]"
3,early_life,7,"[RO-100003, RO-100008, RO-100014, RO-100016, RO-100020, RO-100024]"
4,customer_distress,3,"[RO-100010, RO-100016, RO-100024]"
5,fleet_signal,3,"[RO-100006, RO-100016, RO-100017]"


**Finding.** Five text-observable indicator families occur in the data, plus one derived
from structured mileage. Design decision this locks:

- severity is an **ordinal enum** (`low | medium | high | critical`) **assigned by the LLM**,
  but the schema *also* carries the boolean indicator flags (`safety_related`, `repeat_visit`, ...)
  as separate fields. The flags are auditable and deterministic-checkable; the ordinal is judgment.
  An analyst can sort by the ordinal but *trust* the flags.
- anchor definitions (documented in the prompt): `critical` = safety-relevant or vehicle-disabling;
  `high` = repeat visit, early-life failure, or fleet signal; `medium` = confirmed defect repaired
  routinely; `low` = wear/maintenance/no-defect.

## 8. Recurrence signals

**What we're extracting here:** repeat-visit markers deserve their own field because recurrence
is the single strongest "emerging defect" amplifier — and it's mineable only from text
(this dataset has no VIN linking visits; that's a documented data-limitation / stakeholder
question: *"can we get VINs to track repeat visits deterministically?"*).

In [13]:
# Two DISTINCT signals — do not conflate:
# repeat_visit = service HISTORY (prior visits for same concern)
# intermittent = fault BEHAVIOR (comes and goes) -> repeat-visit *candidate*, captured by resolution=monitoring
pat_repeat = re.compile(r'\b\d+(?:st|nd|rd|th)\s+visit|repeat visit|again\b|reoccur', re.I)
pat_intermittent = re.compile(r'intermittent|sometimes|randomly', re.I)

print("REPEAT-VISIT evidence:")
display(df[df['technician_note'].str.contains(pat_repeat)][['note_id', 'model', 'technician_note']])
print("INTERMITTENT-FAULT evidence (distinct signal; note overlap with NFF):")
display(df[df['technician_note'].str.contains(pat_intermittent)][['note_id', 'model', 'technician_note']])
# Known precision limits (why the LLM extractor owns the per-note decision, not regex):
# - bare 'twice' excluded: "rebooted twice during road test" (RO-100001) is within-visit
#   symptom frequency, not a repeat visit
# - 'again' can be within-note emphasis; ordinal pattern misses "back for the same issue"

REPEAT-VISIT evidence:


,note_id,model,technician_note
7,RO-100008,Accord,customer reports vehicle hesitates/stumbles on accel then CEL flashes. Could not duplicate. NFF. advised return if r...
9,RO-100010,CR-V,"infotainment reboot AGAIN, 3rd visit for same. cust frustrated. installed updated software and reflashed audio/nav. ..."
12,RO-100013,Civic,"battery dead again, cust jump started twice this week. tested battery weak + parasitic draw ~120mA over spec. suspec..."


INTERMITTENT-FAULT evidence (distinct signal; note overlap with NFF):


,note_id,model,technician_note
0,RO-100001,CR-V,Cust states infotainment screen goes black and reboots randomly while driving. Verified concern - display rebooted t...
14,RO-100015,Odyssey,"sliding door pass side intermittent - sometimes stops halfway. lubed track and adjusted. will monitor, possible motor."
19,RO-100020,Pilot,"warning - 'collision system unavailable' intermittent. cleaned front radar sensor area, no obstruction found. reflas..."
20,RO-100021,CR-V,"no start intermittent, cranks no start when hot. tested low fuel pressure. replaced fuel pump assembly. OOW cust pay."
28,RO-100029,Accord,"intermittent no-crank, single click. starter and battery test ok. found loose ground cable at engine block, cleaned ..."
29,RO-100030,Civic,"AC blows warm intermittently then cold. suspect compressor clutch cycling. could not duplicate today, cold at counte..."


## 9. Semantic consistency check — design note (implemented as V6 in Part 2)

**What this step is for:** the keyword→subsystem map (step 4) is brittle by construction — and partially *circular* (the keywords were authored from these same notes). Before locking the subsystem enum, it should be checked by a mechanism with no knowledge of our keywords.

**Design decision — how, exactly:** an earlier draft of this step proposed zero-shot NLI classification (`facebook/bart-large-mnli`). Rejected, for three reasons:
1. **Domain shorthand is out-of-distribution** for general NLI models — OOW / NFF / BO / R&R score near chance.
2. **Uncalibrated scores** — thresholds are unpickable on 30 unlabeled notes.
3. **Cost** — ~1.6GB of weights for a check that a 90MB embedding model does better.

**Adopted instead (V6, Part 2):** embedding similarity — MiniLM scores each note against per-enum `semantic_descriptions` that live in `schema_spec.yaml` (category *definitions*, authored via LLM-assisted inspection — not patterns fitted to notes). Regex keeps precision on shorthand; embeddings supply paraphrase recall and break the lexicon circularity at the meaning level. Output is a disagreement report, advisory only — V1–V5 remain the deterministic gate.

**At 10K notes this becomes the schema drift detector:** rising regex↔semantic disagreement rate, or a rising semantic-coverage floor / 'other' rate ⇒ the enum no longer fits the data ⇒ trigger schema-locked extension mode (Part 3b).

In [14]:
# Implementation lives in Part 2, V6 (hybrid regex + semantic).
# Kept here as the design record: zero-shot NLI was considered and rejected
# (OOD shorthand, uncalibrated scores, 1.6GB weights) in favor of
# MiniLM embeddings vs semantic_descriptions from schema_spec.yaml.
print("See V6 in Part 2 — hybrid regex + semantic signal detection.")

See V6 in Part 2 — hybrid regex + semantic signal detection.


In [15]:
# NER note: generic NER models (e.g. dslim/bert-base-NER) tag person/org/location —
# vehicle components are not in their label space, so they add nothing here.
# Component-mention mining = noun chunks (step 4) + V6 semantic layer.
# Production seam: map component_mention onto a canonical parts taxonomy
# (retrieval against Honda's internal parts catalog), not open-domain NER.
print("Generic NER skipped by design — see comment.")

Generic NER skipped by design — see comment.


## 10. Schema assembly — the artifact

Everything above converges here. Each field below traces to a numbered step
(`provenance`), which is exactly the argument structure for the README and the follow-up
interview: *no field exists because it seemed nice; every field exists because the data
demanded it.*

Output: `schema_spec.yaml` (human-readable contract, with provenance) and
`extraction_schema.json` (JSON Schema the LLM's structured output must validate against).

In [16]:
import yaml, json

schema_spec = {
    'schema_version': '0.1.0',
    'derived_from': 'repair_notes_sample.csv (n=30), EDA notebook 01',
    'identity_passthrough': {
        'description': 'Trusted structured fields. NEVER re-extracted by the LLM (step 1).',
        'fields': ['note_id', 'date', 'model', 'model_year', 'mileage'],
    },
    'extracted_fields': {
        'complaint_summary': {
            'type': 'string', 'provenance': 'step 2',
            'description': 'One-sentence normalized customer complaint.'},
        'component_mention': {
            'type': 'string', 'provenance': 'step 4 (pass A)',
            'description': 'Free-text failing component as written; generalization seam -> '
                           'maps to canonical parts taxonomy in production.'},
        'subsystem': {
            'type': 'enum', 'provenance': 'step 4 (pass B), validated step 9',
            'values': list(SUBSYSTEM_KEYWORDS.keys()) + ['other'],
            'description': 'Locked aggregation axis. "other" is the drift release valve.'},
        'warranty_status': {
            'type': 'enum', 'provenance': 'step 5',
            'values': ['covered', 'denied', 'undetermined', 'not_applicable'],
            'description': 'Spectrum, not boolean: ~2/3 of notes carry no explicit determination.'},
        'denial_reason': {
            'type': 'enum|null', 'provenance': 'step 5',
            'values': ['wear_item', 'out_of_warranty', 'external_cause', 'customer_caused']},
        'resolution_status': {
            'type': 'enum', 'provenance': 'step 6',
            'values': ['repaired', 'declined', 'escalated', 'monitoring', 'nff',
                       'parts_pending', 'no_defect', 'unclear']},
        'severity': {
            'type': 'enum (ordinal)', 'provenance': 'step 7',
            'values': ['low', 'medium', 'high', 'critical'],
            'description': 'LLM judgment with documented anchors; audited via flags below.'},
        'severity_flags': {
            'type': 'object of booleans', 'provenance': 'steps 7-8',
            'fields': ['safety_related', 'vehicle_disabled', 'repeat_visit',
                       'customer_distress', 'fleet_signal'],
            'description': 'Deterministically cross-checkable evidence behind the ordinal.'},
        'confidence': {
            'type': 'enum', 'values': ['high', 'medium', 'low'], 'provenance': 'design',
            'description': 'LLM self-report; low-confidence rows land in the analyst review queue.'},
        'evidence_quote': {
            'type': 'string', 'provenance': 'design (auditability)',
            'description': 'Verbatim span from the note supporting subsystem+severity; '
                           'must be a substring of the note (deterministic validation).'},
    },
    'uncertainty_contract': (
        'Any field the note does not support MUST be undetermined/unclear/null - never guessed. '
        'Rows with confidence=low or validation failures render in a needs-review queue, '
        'excluded from defect aggregations by default.'),
    'stakeholder_questions': [
        'Can notes be linked by VIN to detect repeat visits deterministically? (step 8)',
        'Does an internal canonical parts taxonomy exist to replace the mined subsystem enum?',
        'Is there an official abbreviation glossary to seed the lexicon? (step 3)',
    ],
}

with open('schema_spec.yaml', 'w') as f:
    yaml.safe_dump(schema_spec, f, sort_keys=False, width=100)
print(open('schema_spec.yaml').read()[:1500], '...')

schema_version: 0.1.0
derived_from: repair_notes_sample.csv (n=30), EDA notebook 01
identity_passthrough:
  description: Trusted structured fields. NEVER re-extracted by the LLM (step 1).
  fields:
  - note_id
  - date
  - model
  - model_year
  - mileage
extracted_fields:
  complaint_summary:
    type: string
    provenance: step 2
    description: One-sentence normalized customer complaint.
  component_mention:
    type: string
    provenance: step 4 (pass A)
    description: Free-text failing component as written; generalization seam -> maps to canonical parts
      taxonomy in production.
  subsystem:
    type: enum
    provenance: step 4 (pass B), validated step 9
    values:
    - infotainment_electronics
    - hvac
    - powertrain
    - electrical_battery
    - body_doors
    - chassis_suspension
    - brakes
    - adas_safety
    - tires_wheels
    - other
    description: Locked aggregation axis. "other" is the drift release valve.
  warranty_status:
    type: enum
    proven

In [17]:
# Enrich the emitted spec with semantic_descriptions — consumed by V6 (hybrid layer)
# and by the production discovery module. Lives HERE so every notebook run re-emits
# the enriched spec (an earlier out-of-band enrichment was clobbered by re-runs — bug).
import yaml as _yaml
_spec = _yaml.safe_load(open("schema_spec.yaml"))
SEMANTIC_DESCRIPTIONS = {
 "subsystem": {
  "infotainment_electronics": "problems with the touchscreen, display, audio, navigation, bluetooth, or backup camera display electronics",
  "hvac": "air conditioning or heating problems: weak or warm airflow, refrigerant leaks, compressor, condenser, or evaporator issues",
  "powertrain": "engine, transmission, fuel delivery, or hybrid drive problems: misfires, oil leaks, shuddering, hesitation, no-start from fuel or ignition, check engine codes",
  "electrical_battery": "battery, charging, wiring, grounds, sensors, or starter electrical problems: dead battery, parasitic drain, damaged harness, no-crank",
  "body_doors": "body, doors, or interior trim problems: sliding doors not opening or closing, rattles, loose trim",
  "chassis_suspension": "suspension, steering, or alignment problems: pulling, clunks, loose steering, worn bushings or control arms",
  "brakes": "brake problems: squealing, worn pads, scored rotors",
  "adas_safety": "driver-assistance safety system problems: collision warning unavailable, radar sensor faults, ADAS module issues",
  "tires_wheels": "tire, wheel, wheel-bearing, or tire-pressure-sensor problems: uneven wear, humming bearing noise, TPMS warnings",
  "other": "a problem that does not fit any other vehicle subsystem category",
 },
 "warranty_status": {
  "covered": "the repair was performed under warranty at no cost to the customer",
  "denied": "the repair was determined not to be covered by warranty; customer pays or claim refused",
  "undetermined": "no warranty decision was reached: fault not reproduced, repair declined or pending, case escalated or being monitored",
  "not_applicable": "no defect exists so no warranty question arises: normal operation explained, or damage from an external cause",
 },
 "denial_reason": {
  "wear_item": "a normal wear-and-tear item such as brake pads or tires, not a defect",
  "out_of_warranty": "warranty coverage has expired by time or mileage",
  "external_cause": "damage caused by something outside the vehicle, such as animals, accidents, or environment",
  "customer_caused": "caused by customer actions or aftermarket modifications",
 },
 "resolution_status": {
  "repaired": "the problem was fixed: a part replaced, adjusted, reprogrammed, cleaned, or secured, and verified working",
  "declined": "a repair was recommended but the customer declined it",
  "escalated": "the case was escalated to a technical support line or field engineering, awaiting guidance",
  "monitoring": "no definitive fix; customer advised to monitor and return if the problem recurs",
  "nff": "the reported problem could not be reproduced or verified; no fault found",
  "parts_pending": "repair identified but waiting on parts that are ordered or backordered",
  "no_defect": "inspection found normal operation; customer was educated, no repair needed",
  "unclear": "the note does not make the outcome clear",
 },
 "severity_flags": {
  "safety_related": "the problem affects a safety system or creates a safety risk: brakes, collision avoidance, backup camera, loss of control",
  "vehicle_disabled": "the vehicle would not start or could not be driven",
  "repeat_visit": "the customer has already brought the vehicle in before for this same problem",
  "customer_distress": "the customer is described as frustrated, upset, or losing confidence in the vehicle",
  "fleet_signal": "the note references a wider pattern: a technical service bulletin, tech line, or district reps collecting data",
  "intermittent": "the fault comes and goes rather than being constantly present: happens sometimes, randomly, or could not be reproduced on demand",
 },
}
for f, vals in SEMANTIC_DESCRIPTIONS.items():
    if f in _spec["extracted_fields"]:
        _spec["extracted_fields"][f]["semantic_descriptions"] = vals
_yaml.safe_dump(_spec, open("schema_spec.yaml", "w"), sort_keys=False, allow_unicode=True, width=110)
print("schema_spec.yaml enriched with semantic_descriptions:", list(SEMANTIC_DESCRIPTIONS))

schema_spec.yaml enriched with semantic_descriptions: ['subsystem', 'warranty_status', 'denial_reason', 'resolution_status', 'severity_flags']


In [18]:
# JSON Schema: the literal contract for LLM structured output validation.
ENUMS = {
    'subsystem': list(SUBSYSTEM_KEYWORDS.keys()) + ['other'],
    'warranty_status': ['covered', 'denied', 'undetermined', 'not_applicable'],
    'denial_reason': ['wear_item', 'out_of_warranty', 'external_cause', 'customer_caused', None],
    'resolution_status': ['repaired', 'declined', 'escalated', 'monitoring', 'nff',
                          'parts_pending', 'no_defect', 'unclear'],
    'severity': ['low', 'medium', 'high', 'critical'],
    'confidence': ['high', 'medium', 'low'],
}

json_schema = {
    '$schema': 'https://json-schema.org/draft/2020-12/schema',
    'title': 'RepairNoteExtraction',
    'type': 'object',
    'additionalProperties': False,
    'required': ['complaint_summary', 'component_mention', 'subsystem', 'warranty_status',
                 'resolution_status', 'severity', 'severity_flags', 'confidence',
                 'evidence_quote'],
    'properties': {
        'complaint_summary': {'type': 'string', 'maxLength': 200},
        'component_mention': {'type': 'string', 'maxLength': 80},
        'subsystem': {'enum': ENUMS['subsystem']},
        'warranty_status': {'enum': ENUMS['warranty_status']},
        'denial_reason': {'enum': ENUMS['denial_reason']},
        'resolution_status': {'enum': ENUMS['resolution_status']},
        'severity': {'enum': ENUMS['severity']},
        'severity_flags': {
            'type': 'object',
            'additionalProperties': False,
            'required': ['safety_related', 'vehicle_disabled', 'repeat_visit',
                         'customer_distress', 'fleet_signal'],
            'properties': {k: {'type': 'boolean'} for k in
                           ['safety_related', 'vehicle_disabled', 'repeat_visit',
                            'customer_distress', 'fleet_signal']},
        },
        'confidence': {'enum': ENUMS['confidence']},
        'evidence_quote': {'type': 'string', 'maxLength': 300},
    },
}

with open('extraction_schema.json', 'w') as f:
    json.dump(json_schema, f, indent=2)
print(json.dumps(json_schema, indent=2)[:900], '...')

{
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "RepairNoteExtraction",
  "type": "object",
  "additionalProperties": false,
  "required": [
    "complaint_summary",
    "component_mention",
    "subsystem",
    "warranty_status",
    "resolution_status",
    "severity",
    "severity_flags",
    "confidence",
    "evidence_quote"
  ],
  "properties": {
    "complaint_summary": {
      "type": "string",
      "maxLength": 200
    },
    "component_mention": {
      "type": "string",
      "maxLength": 80
    },
    "subsystem": {
      "enum": [
        "infotainment_electronics",
        "hvac",
        "powertrain",
        "electrical_battery",
        "body_doors",
        "chassis_suspension",
        "brakes",
        "adas_safety",
        "tires_wheels",
        "other"
      ]
    },
    "warranty_status": {
      "enum": [
        "covered",
        "d ...


## Wrap-up: what this module is, in one paragraph

A deterministic, LLM-free pipeline that interrogates the raw notes and emits the extraction
contract. On 30 notes it runs in seconds; on 10K notes it runs as a batch job that refreshes
the lexicon, re-validates the taxonomy (keyword vs. semantic-description agreement, V6), and detects schema
drift ('unmapped'/'other' rates rising). The LLM extraction stage *consumes* the artifacts
produced here — glossary in the prompt, enums in the structured-output schema, regex families
as post-hoc validators — and never has to invent structure on the fly.

---
# Part 2 — Schema Validation
Validates `extraction_schema.json` (authored from Part 1 evidence) against the full corpus.
**Purpose:** validate the hand-derived extraction schema (`extraction_schema.json`, JSON Schema draft 2020-12 — the exact contract the LLM extractor will be held to) against the full 30-note corpus using **zero LLM calls** — pure pandas + regex lexicons (optional spaCy at the end).

**Checks performed:**
1. **V1 — Component coverage:** every note maps to ≥1 `subsystem` enum value.
2. **V2 — Warranty signal coverage:** every note carries evidence for one of the 4 warranty states; silent notes are shown to justify the `undetermined` default.
3. **V3 — Flag coverage:** repeat-visit / NFF / declined / safety / escalation signals are detectable.
4. **V4 — Orphan vocabulary:** frequent terms not claimed by any lexicon → candidate schema gaps.
5. **V5 — Schema↔lexicon consistency:** every lexicon category exists in the schema enums.

**Role in the project:** this is the *schema validation* module. Hardening path: at scale it grows into schema *generation* via budgeted LLM-based EDA (stratified sampling → embed+cluster → LLM labels cluster representatives only → versioned schema artifact). See architecture notes.

**Inputs expected next to this notebook:** `repair_notes_sample.csv`, `extraction_schema.json`.

In [19]:
import json, re
from pathlib import Path
from collections import Counter
import pandas as pd

def find(fname):
    for p in [Path(fname), Path("..")/fname, Path("../schema")/fname, Path("schema")/fname]:
        if p.exists(): return p
    raise FileNotFoundError(fname)

df = pd.read_csv(find("repair_notes_sample.csv"))
schema = json.loads(find("extraction_schema.json").read_text())
print(f"{len(df)} notes | schema: {schema['title']} ({len(schema['properties'])} extracted fields)")
df.head(3)

30 notes | schema: RepairNoteExtraction (10 extracted fields)


,note_id,date,model,model_year,mileage,technician_note
0,RO-100001,2026-05-28,CR-V,2022,14230,Cust states infotainment screen goes black and reboots randomly while driving. Verified concern - display rebooted t...
1,RO-100002,2026-04-08,Accord,2019,68540,"Front brakes squealing. Pads worn to 2mm, rotors scored. R&R front pads and rotors. Cust declined rear at this time...."
2,RO-100003,2026-06-02,Civic,2023,9110,A/C not blowing cold. Recovered refrigerant - system low. Found leak at condenser. Replaced condenser under warranty...


## Corpus profile
Shape of the data the schema must cover: models, years, dates, note length, shorthand density.

In [20]:
print("Models:", dict(df.model.value_counts()))
print("Years :", dict(df.model_year.value_counts().sort_index()))
print("Dates :", df.date.min(), "->", df.date.max())
print("Mileage:", df.mileage.min(), "-", df.mileage.max())
lens = df.technician_note.str.split().str.len()
print(f"Note length (words): min={lens.min()} median={lens.median():.0f} max={lens.max()}")

ABBREVIATIONS = ["nff","oow","r&r","dtc","tsb","cel","bo","lca","atf","tpms","adas","cust","op check","est","assy","lf","rf","lh"]
abbr = {a: int(df.technician_note.str.lower().str.contains(re.escape(a), regex=True).sum()) for a in ABBREVIATIONS}
print("Shorthand density (notes containing):", {k:v for k,v in sorted(abbr.items(), key=lambda x:-x[1]) if v})

Models: {'CR-V': np.int64(6), 'Civic': np.int64(6), 'Accord': np.int64(5), 'Pilot': np.int64(4), 'Odyssey': np.int64(3), 'Ridgeline': np.int64(2), 'HR-V': np.int64(2), 'Accord Hybrid': np.int64(1), 'CR-V Hybrid': np.int64(1)}
Years : {2018: np.int64(2), 2019: np.int64(4), 2020: np.int64(4), 2021: np.int64(5), 2022: np.int64(6), 2023: np.int64(6), 2024: np.int64(3)}
Dates : 2026-04-08 -> 2026-06-22
Mileage: 2100 - 101300
Note length (words): min=13 median=19 max=34
Shorthand density (notes containing): {'cust': 16, 'est': 9, 'bo': 8, 'nff': 5, 'oow': 2, 'r&r': 2, 'dtc': 2, 'tsb': 2, 'cel': 2, 'op check': 2, 'lf': 2, 'rf': 2, 'lca': 1, 'atf': 1, 'tpms': 1, 'adas': 1, 'assy': 1, 'lh': 1}


## Lexicons
Keyword evidence → schema category. These are **validation instruments, not the extractor** — deliberately high-recall and brittle. Where they fail (conflicts, silence) is exactly the evidence that keyword rules don't suffice and an LLM extractor is justified.

In [21]:
SUBSYSTEM_LEXICON = {
    "infotainment_electronics": ["infotainment","screen","display","audio","nav","bluetooth","reboot","backup camera"],
    "hvac": ["a/c"," ac ","refrigerant","condenser","evap","compressor","recharge","blows warm","not blowing cold","vents"],
    "brakes": ["brake","pads","rotors"],
    "powertrain": ["oil leak","valve cover","misfire","ignition coil","spark plug","cranks no start","no-crank","no start","hesitat","cel","check engine","p0301","p0420","catalyst","trans ","torque converter","atf","shudder","fuel pump","fuel pressure","regen braking","hybrid operation"],
    "electrical_battery": ["battery","parasitic draw","ground cable","wiring harness","starter"],
    "body_doors": ["sliding door","door control motor","trim clip","rattle","glove box"],
    "chassis_suspension": ["control arm","bushing","alignment","steering","clunk","pulls"],
    "tires_wheels": ["tire wear","wheel bearing","hub bearing","tpms"],
    "adas_safety": ["collision system","radar sensor","adas","backup camera"],
}
WARRANTY_LEXICON = {
    "denied":   ["not warranty","out of warranty","oow","cust pay","wear item","external cause","insurance","cust approved est"],
    "covered":  ["under warranty","warranty replaced","warranty -","warranty."],
    "undetermined": ["nff","could not duplicate","declined","escalated","awaiting","monitor","advised return","parts bo"],
}
FLAG_LEXICON = {   # keys = severity_flags in the schema, + resolution signals
    "repeat_visit": ["again","visit","reoccur","repeat"],   # coarse presence-check; ordinal regex in Part 1 is stricter
    "safety_related": ["backup camera","collision system","safety concern","cel flashes","no start","no-crank","brakes"],
    "customer_distress": ["frustrated","very upset","concerned"],
    "vehicle_disabled": ["no start","no-crank","cranks no start","dead"],
    "fleet_signal": ["district reps","tech line","tsb","collecting data"],
    "nff_signal": ["nff","could not duplicate"],
    "intermittent_signal": ["intermittent","sometimes","randomly"],   # fault behavior, NOT repeat_visit
    "declined_signal": ["declined"],
}
def hits(text, kws):
    t = " " + text.lower() + " "
    return [k for k in kws if k in t]

## V1 — Component coverage
Every note must map to ≥1 `subsystem`. Multi-matches are fine (a note can touch several systems); the extractor picks the *primary* one. Zero matches = schema gap.

In [22]:
v1_rows, unmapped = [], []
for _, r in df.iterrows():
    matched = sorted({s for s, kws in SUBSYSTEM_LEXICON.items() if hits(r.technician_note, kws)})
    if not matched: unmapped.append(r.note_id)
    v1_rows.append({"note_id": r.note_id, "systems": matched or "!! UNMAPPED"})
v1 = pd.DataFrame(v1_rows)
print(f"Unmapped: {unmapped or 'NONE — 100% coverage'}")
v1

Unmapped: NONE — 100% coverage


,note_id,systems
0,RO-100001,[infotainment_electronics]
1,RO-100002,[brakes]
2,RO-100003,[hvac]
3,RO-100004,[body_doors]
4,RO-100005,[powertrain]
5,RO-100006,[infotainment_electronics]
6,RO-100007,[powertrain]
7,RO-100008,[powertrain]
8,RO-100009,[tires_wheels]
9,RO-100010,[infotainment_electronics]


## V2 — Warranty signal coverage
Denied-phrases are **masked out of the text first**, then covered-phrases scanned — otherwise `"not warranty."` false-matches the covered pattern `"warranty."` (a bug the first pass of this notebook actually had; kept here as evidence of keyword brittleness).

Notes with **no signal at all** are the empirical justification for the `undetermined` enum value: the LLM must be allowed to say "the note doesn't state a decision".

In [23]:
def warranty_signal(text):
    t = " " + text.lower() + " "
    den = [k for k in WARRANTY_LEXICON["denied"] if k in t]
    for k in den: t = t.replace(k, " ")          # mask denied spans before scanning covered
    cov = [k for k in WARRANTY_LEXICON["covered"] if k in t]
    und = [k for k in WARRANTY_LEXICON["undetermined"] if k in t]
    if cov and den: return "CONFLICT", cov+den
    if cov: return "covered", cov
    if den: return "denied", den
    if und: return "undetermined(signal)", und
    return "silent->undetermined", []

v2 = pd.DataFrame([{"note_id": r.note_id,
                    "signal": (s := warranty_signal(r.technician_note))[0],
                    "evidence": s[1]} for _, r in df.iterrows()])
print(v2.signal.value_counts().to_string(), "\n")
v2

signal
undetermined(signal)    11
silent->undetermined     8
covered                  6
denied                   5 



,note_id,signal,evidence
0,RO-100001,undetermined(signal),"[escalated, awaiting]"
1,RO-100002,denied,"[not warranty, wear item]"
2,RO-100003,covered,[under warranty]
3,RO-100004,undetermined(signal),[parts bo]
4,RO-100005,undetermined(signal),[monitor]
5,RO-100006,silent->undetermined,[]
6,RO-100007,undetermined(signal),[declined]
7,RO-100008,undetermined(signal),"[nff, could not duplicate, advised return]"
8,RO-100009,silent->undetermined,[]
9,RO-100010,silent->undetermined,[]


## V3 — Flag coverage
Boolean flags in the schema (`repeat_visit`, `could_not_duplicate`, `customer_declined`, `safety_flag`) plus escalation signals — confirm each has detectable evidence in the corpus (a flag no note ever triggers is schema dead weight).

In [24]:
v3 = pd.DataFrame([{ "note_id": r.note_id,
    **{flag: bool(hits(r.technician_note, kws)) for flag, kws in FLAG_LEXICON.items()}}
    for _, r in df.iterrows()])
print("Notes triggering each flag:")
print(v3.drop(columns="note_id").sum().to_string())
v3[v3.drop(columns="note_id").any(axis=1)]

Notes triggering each flag:
repeat_visit           3
safety_related         6
customer_distress      3
vehicle_disabled       3
fleet_signal           4
nff_signal             5
intermittent_signal    6
declined_signal        2


,note_id,repeat_visit,safety_related,customer_distress,vehicle_disabled,fleet_signal,nff_signal,intermittent_signal,declined_signal
0,RO-100001,False,False,False,False,True,False,True,False
1,RO-100002,False,True,False,False,False,False,False,True
5,RO-100006,False,False,False,False,True,False,False,False
6,RO-100007,False,False,False,False,False,False,False,True
7,RO-100008,True,True,False,False,False,True,False,False
9,RO-100010,True,False,True,False,False,False,False,False
12,RO-100013,True,False,False,True,False,False,False,False
13,RO-100014,False,False,False,False,False,True,False,False
14,RO-100015,False,False,False,False,False,False,True,False
15,RO-100016,False,False,True,False,True,False,False,False


## V4 — Orphan vocabulary
Frequent terms not claimed by any lexicon. Reviewed manually: are these schema gaps (new component/status the schema can't represent) or just descriptive filler? At scale this cell is what the budgeted-LLM EDA replaces.

In [25]:
all_kw = {k.strip() for kws in [*SUBSYSTEM_LEXICON.values(), *WARRANTY_LEXICON.values(), *FLAG_LEXICON.values()] for k in kws}
STOP = set("the a an and or to of in on at for w with cust customer states state found is not no ok now this that after today when time all per side test road front rear during once".split())
tok = Counter(t for note in df.technician_note
                for t in re.findall(r"[a-z][a-z/&-]{2,}", note.lower())
                if t not in STOP and not any(t in kw or kw in t for kw in all_kw))
pd.DataFrame(tok.most_common(20), columns=["orphan_term","count"])

,orphan_term,count
0,light,3
1,tested,3
2,repair,3
3,updated,3
4,reflashed,3
5,failed,3
6,loose,3
7,verified,2
8,twice,2
9,worn,2


## V5 — Schema ↔ lexicon consistency
Every lexicon category must be a legal schema enum value (guards against the validation instrument and the contract drifting apart).

In [26]:
props = schema["properties"]
schema_subsystems = set(props["subsystem"]["enum"])
schema_flags = set(props["severity_flags"]["properties"])
schema_warranty = set(props["warranty_status"]["enum"])
schema_resolution = set(props["resolution_status"]["enum"])
errors = []
if not set(SUBSYSTEM_LEXICON) <= schema_subsystems:
    errors.append(f"lexicon subsystems not in schema: {set(SUBSYSTEM_LEXICON) - schema_subsystems}")
for w in ["covered","denied","undetermined","not_applicable"]:
    if w not in schema_warranty: errors.append(f"warranty state missing: {w}")
lex_flags = {f for f in FLAG_LEXICON if not f.endswith("_signal")}
if not lex_flags <= schema_flags:
    errors.append(f"lexicon flags not in schema severity_flags: {lex_flags - schema_flags}")
for rs in ["nff","declined","escalated","monitoring","parts_pending"]:
    if rs not in schema_resolution: errors.append(f"resolution state missing: {rs}")
print("PASS — lexicons consistent with schema" if not errors else "FAIL:\n" + "\n".join(errors))

PASS — lexicons consistent with schema


## Validation report
Machine-readable summary written next to the notebook — the artifact the later `schema_validation` module in the project will reproduce.

In [27]:
report = {
    "schema": schema["title"],
    "n_notes": int(len(df)),
    "v1_component_coverage": {"unmapped": unmapped, "pass": not unmapped},
    "v2_warranty_signals": v2.signal.value_counts().to_dict(),
    "v3_flag_trigger_counts": v3.drop(columns="note_id").sum().astype(int).to_dict(),
    "v4_top_orphans": dict(tok.most_common(10)),
    "v5_schema_lexicon_consistent": not errors,
}
Path("schema_validation_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

{
  "schema": "RepairNoteExtraction",
  "n_notes": 30,
  "v1_component_coverage": {
    "unmapped": [],
    "pass": true
  },
  "v2_warranty_signals": {
    "undetermined(signal)": 11,
    "silent->undetermined": 8,
    "covered": 6,
    "denied": 5
  },
  "v3_flag_trigger_counts": {
    "repeat_visit": 3,
    "safety_related": 6,
    "customer_distress": 3,
    "vehicle_disabled": 3,
    "fleet_signal": 4,
    "nff_signal": 5,
    "intermittent_signal": 6,
    "declined_signal": 2
  },
  "v4_top_orphans": {
    "light": 3,
    "tested": 3,
    "repair": 3,
    "updated": 3,
    "reflashed": 3,
    "failed": 3,
    "loose": 3,
    "verified": 2,
    "twice": 2,
    "worn": 2
  },
  "v5_schema_lexicon_consistent": true
}


## Optional — spaCy noun-phrase scan (no LLM)
Deeper orphan detection than V4: noun chunks not covered by lexicons. Skipped gracefully if the model isn't installed (`python -m spacy download en_core_web_sm`).

In [28]:
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    chunks = Counter()
    for note in df.technician_note:
        for ch in nlp(note.lower()).noun_chunks:
            c = ch.text.strip()
            if len(c) > 3 and not any(k in c for k in all_kw):
                chunks[c] += 1
    print("Noun chunks not covered by lexicons:")
    for c, n in chunks.most_common(15): print(f"  {c}: {n}")
except Exception as e:
    print(f"spaCy skipped ({type(e).__name__}): {e}")

Noun chunks not covered by lexicons:
  warranty: 5
  leak: 2
  spec: 2
  ring: 2
  20 min road test: 1
  no dtcs: 1
  harness connectors: 1
  guidance: 1
  cust: 1
  this time: 1
  beeps: 1
  reverses: 1
  obstruction sensor: 1
  sensor/harness: 1
  parts: 1


## Findings & limitations
**Findings from this corpus (v0.1.0):**
- 100% of notes map to a `subsystem` — the 13-value enum is sufficient for this sample.
- Warranty distribution is genuinely three-way: explicit covered, explicit denied (4 distinct reasons), and ~⅓ of notes silent/undecided → validates the 4-state enum over a boolean.
- Repeat-visit, NFF, declined, safety, and escalation signals all occur → every schema flag earns its place.

**Limitations (deliberate — these justify the LLM extractor):**
- Substring matching is brittle: negation ("not warranty"), overlapping spans ("out of warranty -"), and word-boundary noise require masking hacks. Keyword rules can *detect signal presence* for validation, but cannot *decide* per-note fields reliably.
- No semantic understanding: "explained normal hybrid operation" reads as no-defect to a human, invisible to keywords.

**Hardening path:** at 10K+ notes this notebook becomes a `schema_validation` module and grows a schema-*generation* mode: stratified sampling → embeddings + clustering (no LLM) → budgeted LLM labeling of cluster representatives only → convergence stopping rule → versioned schema artifact diffed before adoption.

## V6 — Hybrid signal detection (regex ∪ semantic)
Regex lexicons (V1–V3) are precise on domain shorthand but were authored from this corpus → partially circular, and brittle on paraphrase. This layer breaks the circularity at the *meaning* level:

- **Category definitions** come from `schema_spec.yaml` → `semantic_descriptions` (single source of truth; authored via LLM-assisted inspection, NOT patterns fitted to notes).
- **MiniLM** (`sentence-transformers/all-MiniLM-L6-v2`, ~90MB) embeds every note and every description; cosine similarity scores each note × category.
- **Output = disagreement report, both directions:**
  - *semantic top-1 with no regex hit* → recall gap (paraphrase the lexicon misses)
  - *regex hit ranked low semantically* → precision suspect (e.g. the bare-"twice" class)
- **No hard thresholds** — 30 unlabeled notes can't calibrate one. Ranked tables for human review; at scale, thresholds come from labeled spot-checks.

Guarded: skips gracefully if `sentence-transformers` / the model isn't available. Install locally: `pip install sentence-transformers` (model auto-downloads, or point `HF_HOME` at your packaged copy).

In [29]:
try:
    from sentence_transformers import SentenceTransformer, util
    import yaml
    spec = yaml.safe_load(open(find("schema_spec.yaml")))
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    note_emb = model.encode(list(df.technician_note), normalize_embeddings=True)

    def semantic_scores(desc_map):
        cats = list(desc_map)
        demb = model.encode([desc_map[c] for c in cats], normalize_embeddings=True)
        return cats, util.cos_sim(note_emb, demb).numpy()   # (n_notes, n_cats)

    # ---- 6a. Subsystem: regex vs semantic top-1 ----
    sub_desc = spec["extracted_fields"]["subsystem"]["semantic_descriptions"]
    cats, S = semantic_scores(sub_desc)
    rows = []
    for i, r in df.iterrows():
        regex_hits = sorted({s for s, kws in SUBSYSTEM_LEXICON.items() if hits(r.technician_note, kws)})
        top = S[i].argsort()[::-1][:2]
        rows.append({"note_id": r.note_id,
                     "regex": regex_hits,
                     "semantic_top1": cats[top[0]], "s1": round(float(S[i][top[0]]), 3),
                     "semantic_top2": cats[top[1]], "s2": round(float(S[i][top[1]]), 3),
                     "agree": cats[top[0]] in regex_hits})
    v6a = pd.DataFrame(rows)
    print(f"6a subsystem agreement (semantic top-1 in regex hits): {v6a.agree.sum()}/{len(v6a)}")
    print("\nDISAGREEMENTS — review list (recall gaps or ambiguous notes):")
    display(v6a[~v6a.agree])

    # ---- 6b. Resolution + warranty + flags: same pattern, disagreements only ----
    def hybrid_disagreements(field, regex_map, kind="lexicon"):
        desc = spec["extracted_fields"][field]["semantic_descriptions"]
        cats_f, Sf = semantic_scores(desc)
        out = []
        for i, r in df.iterrows():
            if kind == "lexicon":
                rx = sorted({c for c, kws in regex_map.items() if hits(r.technician_note, kws)})
            else:
                rx = sorted({c for c, pat in regex_map.items() if pat.search(r.technician_note)})
            top1 = cats_f[Sf[i].argmax()]
            if rx and top1 not in rx:
                out.append({"note_id": r.note_id, "regex": rx, "semantic_top1": top1,
                            "score": round(float(Sf[i].max()), 3)})
            elif not rx:
                out.append({"note_id": r.note_id, "regex": "(none)", "semantic_top1": top1,
                            "score": round(float(Sf[i].max()), 3)})
        return pd.DataFrame(out)

    RESOLUTION_LEXICON = {
        "repaired": ["replaced","r&r","repaired","reflashed","reprogrammed","adjusted","secured","recharged","lubed","torqued"],
        "declined": ["declined"], "escalated": ["escalated","awaiting guidance","collecting data"],
        "monitoring": ["monitor","advised return","return if"], "nff": ["nff","could not duplicate","no fault"],
        "parts_pending": [" bo","backorder","ordered"], "no_defect": ["explained normal","no repair needed"],
    }
    print("\n6b resolution — regex↔semantic disagreements + regex-silent notes:")
    display(hybrid_disagreements("resolution_status", RESOLUTION_LEXICON))

    # ---- 6c. Semantic coverage floor: notes far from EVERY subsystem ----
    floor = pd.DataFrame({"note_id": df.note_id, "max_sim_any_subsystem": S.max(axis=1).round(3)}
                        ).sort_values("max_sim_any_subsystem")
    print("\n6c semantic coverage floor — lowest = candidates the schema may not cover:")
    display(floor.head(8))

except ImportError:
    print("sentence-transformers not installed — skipping V6 (hybrid semantic layer).")
    print("To enable: pip install sentence-transformers")
except KeyError as e:
    print(f"semantic_descriptions missing from schema_spec.yaml ({e}) — run the enrichment cell in Part 1 first.")
except OSError as e:
    print(f"MiniLM model unavailable ({e}) — skipping V6. Point HF_HOME at the packaged model or allow download.")

sentence-transformers not installed — skipping V6 (hybrid semantic layer).
To enable: pip install sentence-transformers


---
# Part 3 — Schema Generation (no-LLM discovery mode)
What the `--discover` flag will produce **without any LLM data inspection**: an internal schema definition derived purely from the corpus evidence above (lexicon hits, signal categories), rendered as BOTH artifacts — `generated_extraction_schema.json` (contract) and `generated_schema_spec.yaml` (record) — from one source.

This generated schema could run the entire pipeline (same field skeleton, schema-driven extractor), just **coarser** than the authored one. The diff at the end quantifies what LLM-inspected authoring added.

In [30]:
# One internal definition, derived only from corpus evidence (no LLM)
generated_def = {
    "title": "RepairNoteExtractionGenerated",
    "provenance": "auto-generated from corpus evidence (lexicon-derived, zero LLM calls)",
    "fields": {
        "component_mention": {"type": "string", "maxLength": 80,
            "rationale": "frequent part-noun evidence in corpus"},
        "subsystem": {"type": "enum",
            "values": sorted(SUBSYSTEM_LEXICON) + ["other"],
            "rationale": "clusters of co-occurring component keywords (V1 evidence)"},
        "warranty_status": {"type": "enum",
            "values": ["covered", "denied", "undetermined", "not_applicable"],
            "rationale": "three-way signal split observed in V2 (incl. 8 silent notes)"},
        "denial_reason": {"type": "enum",
            "values": ["wear_item", "out_of_warranty", "external_cause", "customer_caused", None],
            "rationale": "distinct denial phrases in corpus"},
        "resolution_status": {"type": "enum",
            "values": ["repaired", "declined", "escalated", "monitoring", "nff", "parts_pending", "unclear"],
            "rationale": "outcome keywords: declined/escalated/monitor/nff/parts bo"},
        "severity": {"type": "enum", "values": ["low", "medium", "high"],
            "rationale": "coarse 3-level default; corpus keywords cannot anchor levels"},
        "severity_flags": {"type": "object",
            "properties": sorted(f for f in FLAG_LEXICON if not f.endswith("_signal")),
            "rationale": "flag keywords each trigger >=3 times (V3)"},
    },
}
print("generated definition:", list(generated_def["fields"]))

generated definition: ['component_mention', 'subsystem', 'warranty_status', 'denial_reason', 'resolution_status', 'severity', 'severity_flags']


In [31]:
# Render BOTH artifacts from the one definition
import yaml, json

def to_json_schema(d):
    props, req = {}, []
    for name, f in d["fields"].items():
        req.append(name)
        if f["type"] == "string":
            props[name] = {"type": "string", "maxLength": f["maxLength"]}
        elif f["type"] == "enum":
            props[name] = {"enum": f["values"]}
        elif f["type"] == "object":
            props[name] = {"type": "object", "additionalProperties": False,
                           "required": f["properties"],
                           "properties": {p: {"type": "boolean"} for p in f["properties"]}}
    return {"$schema": "https://json-schema.org/draft/2020-12/schema",
            "title": d["title"], "type": "object",
            "additionalProperties": False, "required": req, "properties": props}

def to_yaml_spec(d):
    return {"schema_version": "generated-0.0.1", "provenance": d["provenance"],
            "identity_passthrough": ["note_id", "date", "model", "model_year", "mileage"],
            "extracted_fields": {n: {k: v for k, v in f.items()} for n, f in d["fields"].items()}}

gen_json = to_json_schema(generated_def)
Path("generated_extraction_schema.json").write_text(json.dumps(gen_json, indent=2))
Path("generated_schema_spec.yaml").write_text(yaml.safe_dump(to_yaml_spec(generated_def), sort_keys=False, allow_unicode=True))
print("wrote generated_extraction_schema.json + generated_schema_spec.yaml")
print(json.dumps(gen_json, indent=1)[:800])

wrote generated_extraction_schema.json + generated_schema_spec.yaml
{
 "$schema": "https://json-schema.org/draft/2020-12/schema",
 "title": "RepairNoteExtractionGenerated",
 "type": "object",
 "additionalProperties": false,
 "required": [
  "component_mention",
  "subsystem",
  "warranty_status",
  "denial_reason",
  "resolution_status",
  "severity",
  "severity_flags"
 ],
 "properties": {
  "component_mention": {
   "type": "string",
   "maxLength": 80
  },
  "subsystem": {
   "enum": [
    "adas_safety",
    "body_doors",
    "brakes",
    "chassis_suspension",
    "electrical_battery",
    "hvac",
    "infotainment_electronics",
    "powertrain",
    "tires_wheels",
    "other"
   ]
  },
  "warranty_status": {
   "enum": [
    "covered",
    "denied",
    "undetermined",
    "not_applicable"
   ]
  },
  "denial_reason": {
   "enum": [
    "wear_item",



## Generated vs authored — what LLM-inspected authoring added

In [32]:
auth_props, gen_props = set(schema["properties"]), set(gen_json["properties"])
print("Fields only in AUTHORED schema:", sorted(auth_props - gen_props))
print("Fields only in GENERATED schema:", sorted(gen_props - auth_props))
for f in sorted(auth_props & gen_props):
    a, g = schema["properties"][f], gen_json["properties"][f]
    ae, ge = a.get("enum"), g.get("enum")
    if ae and ge and set(map(str, ae)) != set(map(str, ge)):
        print(f"\n{f}:")
        print("  authored :", ae)
        print("  generated:", ge)
print("""\nGap summary (the value of LLM inspection):
- complaint_summary, confidence, evidence_quote exist only in the authored schema
  (semantic fields keyword evidence cannot justify)
- severity: authored 4-level with 'critical' + anchored definitions vs generated
  unanchored 3-level default
- resolution: authored adds 'no_defect' (education/normal-operation visits) —
  invisible to keywords, discovered only by reading the notes
- Same skeleton otherwise -> the generated schema could run the pipeline end-to-end,
  just coarser. This is the --discover mode demo.""")

Fields only in AUTHORED schema: ['complaint_summary', 'confidence', 'evidence_quote']
Fields only in GENERATED schema: []

resolution_status:
  authored : ['repaired', 'declined', 'escalated', 'monitoring', 'nff', 'parts_pending', 'no_defect', 'unclear']
  generated: ['repaired', 'declined', 'escalated', 'monitoring', 'nff', 'parts_pending', 'unclear']

severity:
  authored : ['low', 'medium', 'high', 'critical']
  generated: ['low', 'medium', 'high']

Gap summary (the value of LLM inspection):
- complaint_summary, confidence, evidence_quote exist only in the authored schema
  (semantic fields keyword evidence cannot justify)
- severity: authored 4-level with 'critical' + anchored definitions vs generated
  unanchored 3-level default
- resolution: authored adds 'no_defect' (education/normal-operation visits) —
  invisible to keywords, discovered only by reading the notes
- Same skeleton otherwise -> the generated schema could run the pipeline end-to-end,
  just coarser. This is the --d

In [33]:
# Validate the GENERATED schema is itself well-formed and pipeline-usable
import jsonschema
jsonschema.Draft202012Validator.check_schema(gen_json)
example = {"component_mention": "audio/nav unit", "subsystem": "infotainment_electronics",
           "warranty_status": "undetermined", "denial_reason": None,
           "resolution_status": "monitoring", "severity": "medium",
           "severity_flags": {f: False for f in generated_def["fields"]["severity_flags"]["properties"]}}
jsonschema.validate(example, gen_json)
print("generated schema: well-formed JSON Schema 2020-12, example record validates — pipeline-usable")

generated schema: well-formed JSON Schema 2020-12, example record validates — pipeline-usable


## Part 3b — Schema-locked extension mode (production default for `--discover`)
Bootstrap generation (above) is for greenfield only. In production, discovery runs **seeded from the locked schema** and is **monotonic**: it may only ADD enum values and OPTIONAL fields — never remove, rename, retype, or add `required` fields (a new required field would invalidate every previously extracted record in the S3 cache; promoting to required = major version + re-extraction).

Demo: the corpus-evidenced `intermittent` signal (6 notes, overlaps 4/5 NFF notes) proposed as an extension to `severity_flags`, with a machine-checked backward-compatibility proof.

V6 is advisory throughout — V1–V5 remain the deterministic pipeline gate and are unaffected by the semantic layer's presence or absence.

In [34]:
import copy, jsonschema

def extend_schema(locked, additions):
    """Monotonic extension: enum additions + optional field additions only.
    Raises on anything that would remove/rename/retype or add required fields."""
    ext = copy.deepcopy(locked)
    log = []
    for path, add in additions.get("enum_additions", {}).items():
        node = ext["properties"]
        parts = path.split(".")
        for p in parts[:-1]:
            node = node[p]["properties"] if "properties" in node[p] else node[p]
        target = node[parts[-1]]
        assert "enum" in target, f"{path} is not an enum"
        new = [v for v in add if v not in target["enum"]]
        target["enum"] = target["enum"] + new
        log.append(f"enum {path}: +{new}")
    for parent, fields in additions.get("optional_field_additions", {}).items():
        node = ext["properties"][parent] if parent else ext
        for fname, fdef in fields.items():
            assert fname not in node["properties"], f"{fname} already exists"
            node["properties"][fname] = fdef
            # NEVER appended to node["required"] — that is the compat rule
            log.append(f"optional field {parent or '<root>'}.{fname}: added (NOT required)")
    return ext, log

PROPOSED = {
    "optional_field_additions": {
        "severity_flags": {"intermittent": {"type": "boolean"}},
    },
    # example of the other legal move (not applied — none evidenced yet):
    # "enum_additions": {"resolution_status": ["recall_pending"]},
}
extended, changelog = extend_schema(schema, PROPOSED)
print("EXTENSION DIFF (goes to review gate):")
for l in changelog: print(" +", l)

EXTENSION DIFF (goes to review gate):
 + optional field severity_flags.intermittent: added (NOT required)


In [35]:
# Backward-compatibility PROOF: every record valid under the locked schema
# must remain valid under the extended schema (extension is a superset).
jsonschema.Draft202012Validator.check_schema(extended)

old_record = {   # shaped by the LOCKED schema — no 'intermittent' key
    "complaint_summary": "A/C not blowing cold",
    "component_mention": "condenser",
    "subsystem": "hvac",
    "warranty_status": "covered",
    "denial_reason": None,
    "resolution_status": "repaired",
    "severity": "medium",
    "severity_flags": {"safety_related": False, "vehicle_disabled": False,
                       "repeat_visit": False, "customer_distress": False,
                       "fleet_signal": False},
    "confidence": "high",
    "evidence_quote": "Replaced condenser under warranty",
}
jsonschema.validate(old_record, schema)      # valid under locked
jsonschema.validate(old_record, extended)    # MUST stay valid under extended

new_record = copy.deepcopy(old_record)
new_record["severity_flags"]["intermittent"] = True
jsonschema.validate(new_record, extended)    # new records may use the addition
try:
    jsonschema.validate(new_record, schema)  # and are correctly REJECTED by locked
    print("!! locked schema accepted the new flag — additionalProperties leak")
except jsonschema.ValidationError:
    print("compat proof PASSED:")
    print("  old records validate under locked AND extended schema")
    print("  new records validate under extended; rejected by locked (as expected)")
    print("-> extension is a monotonic, backward-compatible superset")

compat proof PASSED:
  old records validate under locked AND extended schema
  new records validate under extended; rejected by locked (as expected)
-> extension is a monotonic, backward-compatible superset
